# Datenquelle NOAA GHCN-Daily: tägliches Wetter je Station

Diese Quelle liefert **Zeitreihen**, keine Ereignisse und keine Referenzgeometrie.
Sie ist die dritte Zeitachse des Projekts und die einzige Quelle, die sich nicht
über `GEOID` anschliesst.

**Pipeline-Stufe:** Quellendokumentation. Die Ingestion selbst liegt in
`weather_pipeline` (`download_noaa.py`).

| | GHCN-Daily |
| --- | --- |
| Inhalt | eine Zeile je Station und Tag, mit Temperatur, Niederschlag, Schnee und rund hundert weiteren Grössen |
| Herausgeber | NOAA, National Centers for Environmental Information |
| Zugriff | eine CSV je Station über HTTPS, **ohne Schlüssel** |
| Änderungsverhalten | die Datei wächst täglich um eine Zeile, wird aber immer ganz ausgeliefert |
| Schlüssel | `STATION` + `DATE`; verbunden wird über **Stadt + Datum** |

`GHCN` steht für *Global Historical Climatology Network*, das `-Daily` für die
Tagesauflösung. Es gibt daneben ein `GHCN-Monthly` mit Monatsmitteln — für eine
Frage nach Kriminalität und Wetter ist die Tagesauflösung die richtige.

## 1. Warum diese Quelle

Die Leitfrage in Variante C (Abschnitt 2.1 der Projektabsprache) fragt nach dem
Einfluss von Wetter auf Kriminalität. Dafür braucht es je Stadt und Tag einen
Messwert, der

1. **denselben Tag** benennt wie die Falldaten,
2. **beide Städte** über dieselbe Methodik abdeckt,
3. **weit genug zurückreicht**, um auch ältere Fälle zu verbinden.

GHCN-Daily erfüllt alle drei Punkte und kostet keinen Schlüssel. Die Alternative
wäre die NOAA-Web-API (`api.weather.gov` oder `ncei.noaa.gov/cdo-web`) gewesen —
beide brauchen einen Token beziehungsweise erlauben nur kleine Zeitfenster je
Abruf. Eine CSV je Station ist dagegen dasselbe Muster wie bei TIGER: eine Datei,
ein Abruf, ein Vollständigkeitsnachweis.

**Wichtig für die Erwartungshaltung:** Wetter ist eine *Ausbaustufe*. Der
Vorschlag in Abschnitt 3 sagt ausdrücklich, erst zu bauen, wenn Variante C
beschlossen ist. Diese Pipeline existiert vorab, damit die Entscheidung an
etwas Sichtbarem getroffen werden kann statt an einer Absichtserklärung.

## 2. Was eine Zeile bedeutet

Eine Zeile ist **eine Station an einem Tag**. Sie trägt drei Arten von Spalten:

**Identität** — wo und wann gemessen wurde: `STATION`, `DATE`, `LATITUDE`,
`LONGITUDE`, `ELEVATION`, `NAME`. Diese Werte sind je Station über die ganze
Reihe konstant; sie stehen trotzdem in jeder Zeile, weil die Quelle dasselbe
Format auch für Sammeldateien über viele Stationen verwendet.

**Messgrössen** — je Grösse eine Wertspalte. Die vier, die das Projekt gepinnt
hat:

| Spalte | Bedeutung | Einheit in der Quelle |
| --- | --- | --- |
| `TMAX` | Tagesmaximum der Temperatur | Zehntelgrad Celsius |
| `TMIN` | Tagesminimum der Temperatur | Zehntelgrad Celsius |
| `PRCP` | Niederschlag | Zehntelmillimeter |
| `SNOW` | Neuschnee | Millimeter |

**Qualitätsflags** — je Messgrösse eine Spalte `<GRÖSSE>_ATTRIBUTES` mit
kommagetrennten Feldern: `M,Q,S` für *measurement*, *quality* und *source*, bei
manchen Grössen zusätzlich die Beobachtungszeit. Der Wert `,,0` heisst: kein
Messhinweis, keine Beanstandung, Quelle `0`. Wie viele Felder es sind, hängt an
der Grösse und nicht an der Zeile — siehe Feststellung 5.8 und Appendix B. Diese
Flags sind Teil der Quelle und wandern unverändert nach Bronze; Abschnitt 3
Frage 3 der Projektabsprache hat das so entschieden.

Was eine Zeile **nicht** ist: kein Stadtmittel. Eine Station steht für eine
Position, hier der jeweilige Flughafen. Ob das für eine ganze Stadt taugt, ist
eine fachliche Frage und keine technische — Abschnitt 7 kommt darauf zurück.

## 3. Die Join-Achse: Datum und Stadt, nicht `GEOID`

Das ist der wichtigste Unterschied zu allen anderen Quellen des Projekts.

Chicago, Detroit und Census treffen sich **räumlich**: Ein Fall hat Koordinaten,
die Koordinaten fallen in ein Tract-Polygon, das Tract hat eine `GEOID`, und
über diese `GEOID` hängen die ACS-Kennzahlen daran. Eine Kette, ein Schlüssel.

Wetter hängt nicht in dieser Kette. Eine Station am Flughafen sagt nichts über
einzelne Tracts aus — Temperatur und Niederschlag variieren innerhalb einer Stadt
kaum, jedenfalls nicht in einer Grössenordnung, die eine Tract-Zuordnung
rechtfertigen würde. Der Anschluss läuft deshalb über

```text
Fall  ──(Tatdatum)──>  Tag  <──(DATE)──  Wetterzeile
Fall  ──(Stadt)─────>  Stadt <──(city)──  Station
```

Zwei Folgen für die Ablage:

1. Der Partitionsschlüssel heisst **`station=`** und nicht `state=`. Eine Station
   gehört zu einer Stadt, nicht zu einem Bundesstaat — und Chicago hätte mit
   O'Hare und Midway zwei Stationen im selben Bundesstaat.
2. Das Manifest nennt die Achse ausdrücklich unter **`join_keys`**. Silver soll
   sie nicht aus dem Pfad erraten müssen.

**Die offene Frage dahinter:** Auf welcher Zeitachse wird verbunden? Die Falldaten
führen Tatzeitpunkt *und* Änderungszeitpunkt, GHCN-Daily nur ein Kalenderdatum in
Ortszeit der Station. Fachlich richtig ist der **Tatzeitpunkt in Ortszeit** — und
genau dieselbe Zeitzonenfrage ist bei Detroit offen (siehe Abschnitt 4.2 und den
Befund zum Daily-Fenster).

## 4. Technischer Zugriff

Eine CSV je Station, ein `GET`, kein Schlüssel, keine Paginierung:

```text
https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/<STATION>.csv
```

| Station | ID | Rolle im Projekt |
| --- | --- | --- |
| Chicago O'Hare International Airport | `USW00094846` | Standardlauf |
| Detroit Metro Airport | `USW00094847` | Standardlauf |
| Chicago Midway Airport | `USW00014819` | hinterlegt, per `--station` zuschaltbar |

Die Stations-IDs folgen einem Schema: `US` für das Land, `W` für *WBAN* (die
Kennung des Wetterdienstes), danach die fünfstellige WBAN-Nummer. Wer weitere
Stationen sucht, findet sie in `ghcnd-stations.txt` im Wurzelverzeichnis desselben
Servers.

**Warum die Flughafenstationen:** Dort messen automatische Stationen rund um die
Uhr, seit Jahrzehnten, mit der geringsten Zahl von Lücken. Innerstädtische
Stationen sind oft Freiwilligenmeldungen mit unvollständigen Reihen.

## 5. Feststellungen aus der Erkundung

Die folgenden Punkte sind nicht aus der Dokumentation abgeschrieben, sondern an
der Quelle geprüft. Mehrere davon widersprechen dem, was Abschnitt 3 der
Projektabsprache angenommen hatte, und jeder einzelne ist Grund für eine
Designentscheidung der Ingestion.

### 5.1 Die Kopfzeile hat über hundert Spalten, nicht elf

Abschnitt 3 listete `STATION, DATE, LATITUDE, LONGITUDE, ELEVATION, NAME, PRCP,
SNOW, SNWD, TMAX, TMIN`. Tatsächlich liefert O'Hare **112** Spalten und Detroit
Metro **116** — unter anderem `TAVG` (Mitteltemperatur), `AWND` (Windmittel),
`ADPT` (Taupunkt), `PGTM` (Zeitpunkt der Spitzenböe) und die Wettertyp-Flags
`WT01` bis `WV20` für Nebel, Gewitter, Rauch und ähnliches.

Genau dafür gibt es das gepinnte Schema plus `_extra`: Die vier beschlossenen
Grössen stehen als eigene Spalten, alles übrige landet unverändert als JSON in
`_extra`. **Nichts wird verworfen** — wer später `TAVG` auswerten will, braucht
keinen neuen Ladelauf, sondern einen Eintrag im Code.

### 5.2 Die Spaltenzahl unterscheidet sich je Station

Das ist keine Schema-Drift über die Zeit wie bei TIGER, sondern eine Eigenschaft
der Station: GHCN-Daily führt je Station nur die Grössen, die dort auch gemessen
werden. Der Schema-Fingerabdruck wird deshalb **je Station** gebildet und nicht
einmal für die Quelle — zwei verschiedene Fingerabdrücke sind hier normal, ein
geänderter Fingerabdruck für dieselbe Station ist das Signal.

### 5.3 Werte sind rechtsbündig aufgefüllter Text in Zehnteln

Die CSV schreibt `"  214"`, nicht `21.4`. Zwei Fallen in einem Wert: die
Leerzeichen und der Faktor zehn. `"  214"` sind **21,4 °C**.

Bronze lässt beides stehen. Abschnitt 4.3 sagt „Werte behalten den Typ der
Quelle; Parsen erst in Silver", und eine CSV hat nur einen Typ: Text. Das
Manifest erklärt die Einheit unter `value_encoding`, damit das in Silver niemand
raten muss.

### 5.4 Die Reihe endet rund vier Tage vor dem Abrufdatum

GHCN-Daily ist ein qualitätsgeprüfter Datensatz, keine Live-Wetterquelle. Zwischen
Messung und Veröffentlichung liegen mehrere Tage. Ein Join auf die jüngsten
Falldaten läuft dort ins Leere — **Eigenschaft der Quelle, kein Fehler der
Ingestion**. Wer aktuelles Wetter braucht, braucht eine andere Quelle.

### 5.5 Die Reihe beginnt 1946, die Temperatur erst 1958

Das ist die unangenehmste Feststellung. O'Hare liefert Zeilen ab **1946-10-09**,
aber `TMAX`, `TMIN`, `PRCP` und `SNOW` sind erst ab **1958-11-01** belegt. Die
Zeilen davor existieren, tragen aber keine der vier Grössen — sie führen nur
`TAVG` und einige Wolkenparameter.

Über die ganze Reihe ist `TMAX` deshalb nur in rund 85 % der Zeilen belegt, und
das ist **kein Qualitätsproblem der jüngeren Jahre**: ab den 1960ern ist die
Belegung lückenlos 100 %. Abschnitt 6.5 zeigt das je Jahrzehnt. Für das Projekt
mit Falldaten ab 2001 ist das folgenlos — aber eine Auswertung, die naiv über die
ganze Reihe zählt, unterschätzt die Datenlage.

### 5.6 Das Qualitätsflag ist fast immer leer — aber nicht immer

Die mittlere Position von `M,Q,S` ist das Qualitätsflag: Ein Wert dort heisst,
dass NOAAs Prüfung den Messwert beanstandet hat.

Für `TMAX`, `TMIN` und `PRCP` ist es an beiden Stationen über die gesamte Reihe
**leer**. Für `SNOW` gilt das nur an Detroit Metro — **O'Hare hat vier Zeilen
mit dem Flag `I`**, zwischen 1999-01-09 und 2011-01-24. `I` steht für einen
fehlgeschlagenen internen Konsistenztest.

Vier von 24'603 ist wenig, aber „durchgängig leer" wäre falsch: Wer das Flag
nicht prüft, übernimmt vier von NOAA selbst beanstandete Werte ungeprüft.
Abschnitt 6.7 zählt deshalb alle vier Grössen aus, statt von zwei auf alle zu
schliessen — genau dieser Fehlschluss stand hier in einer früheren Fassung.

Das Quellenflag an dritter Position variiert dagegen (`0`, `W`, `X`, `1`, bei
Detroits `SNOW` auch `Z`); es benennt nur, aus welchem Meldeweg der Wert
stammt.

### 5.7 `Content-Length` ist da, eine Zeilenzahl nicht

Die Quelle nennt keine erwartete Zeilenzahl, gegen die sich zählen liesse. Sie
nennt aber `Content-Length` und `Last-Modified`. Der Vollständigkeitsnachweis
dieser Quelle ruht deshalb auf drei Beinen:

1. `Content-Length` gegen die empfangenen Bytes,
2. gelesene CSV-Zeilen gegen geschriebene Parquet-Zeilen,
3. die Prüfung, dass **jede** Zeile die angefragte Station nennt.

Der dritte Punkt fängt einen Fehler, der sonst erst in Silver auffällt: eine
falsch einsortierte Partition, wenn die Quelle unter einer ID die Datei einer
anderen Station ausliefert.

### 5.9 Die Quelle ist nicht append-only — sie korrigiert die letzten Tage

Zwei Schnappschüsse im Abstand eines Tages, Zeile für Zeile verglichen:

| | O'Hare | Detroit Metro |
| --- | --- | --- |
| neue Zeilen am Ende | 2 | 2 |
| entfernte Zeilen | 0 | 0 |
| **nachträglich geänderte Zeilen** | **0** | **1** (2026-09-02) |

In Detroit hat die Zeile vom 2.9. nachträglich `SNWD` und `SNWD_ATTRIBUTES`
bekommen, wo vorher leere Felder standen — eine Korrektur einen Tag hinter dem
damaligen Rand der Datei.

**Das ist der Grund, warum jeder Lauf die ganze Datei lädt.** Ein Delta über
`Range`, das nur das Dateiende anhängt, hätte diese Korrektur übersehen. Der
Befund ist auf das Gemessene zu beschränken — ein Intervall, zwei Stationen —,
aber die Richtung ist damit klar: Anhängen genügt nicht.

Nebenbei: die Korrektur betraf `SNWD`, das bei uns in `_extra` liegt. Die vier
gepinnten Grössen waren in diesem Fall nicht betroffen.

### 5.8 Die Flag-Spalten haben je Grösse **und je Station** verschieden viele Felder

Die dritte Annahme, die sich beim Nachzählen nicht gehalten hat. `M,Q,S` ist
nicht immer alles — manche Spalten führen ein viertes Feld mit der
Beobachtungszeit als `HHMM`:

| Spalte | O'Hare | Detroit Metro |
| --- | --- | --- |
| `TMAX_ATTRIBUTES`, `TMIN_ATTRIBUTES` | 3 Felder | 3 Felder |
| `PRCP_ATTRIBUTES` | 4 Felder, Zeit `2400` | 4 Felder, Zeit `2400` |
| `SNOW_ATTRIBUTES` | **4 Felder**, Zeit leer | **3 Felder** |

Innerhalb einer Spalte einer Station ist die Feldzahl konstant, über Stationen
hinweg nicht. Sie lässt sich also je Spalte bestimmen, aber nicht einmal für die
Quelle — und schon gar nicht aus der Dokumentation ableiten.

**Und die Beobachtungszeit kann den literalen Text `null` enthalten.** In Detroit
steht in 919 Zeilen `T,,0,null` statt `T,,0,2400`; ein Beleg aus der Raw-Datei:

```text
"USW00094847","1997-10-02",...,"    0","T,,0,null","    0",",,Z",...
```

Das ist kein leeres Feld, sondern die vier Zeichen `n`, `u`, `l`, `l`. Ein Parser
in Silver, der pauschal drei Felder annimmt oder `null` für einen Fehlwert hält,
verschiebt Positionen beziehungsweise erzeugt eine Zahl aus einem Wort.
Abschnitt 6.7 zählt die Feldzahl deshalb je Spalte mit aus, statt sie zu
unterstellen.

## 6. Die Quelle ansehen

Die folgenden Zellen prüfen die Feststellungen nach. Die Netzzellen brauchen
Internet, die Parquet-Zellen einen geladenen Lauf (Abschnitt 9).

In [ ]:
from pathlib import Path
import collections
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'crime' / 'weather_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'crime' / 'weather_pipeline').exists():
    raise RuntimeError('Projekt-Root mit src/crime/weather_pipeline nicht gefunden')

sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'crime'))

import requests
from weather_pipeline import DEFAULT_STATIONS, STATIONS, download_noaa as dn

session = requests.Session()
session.headers['User-Agent'] = dn.USER_AGENT

print('Dataset   :', dn.DATASET)
print('Basis-URL :', dn.BASE_URL)
print()
for sid, info in STATIONS.items():
    rolle = 'Standardlauf' if info['default'] else 'zuschaltbar'
    print(f"  {sid}  {info['city']:<8} {rolle:<13} {info['name']}")
print()
print('Identitaet:', dn.IDENTITY_COLUMNS)
print('Groessen  :', dn.MEASUREMENT_COLUMNS, '- je zusaetzlich', dn.ATTRIBUTE_SUFFIX)

### 6.1 Die Auslieferungen selbst

Ein `HEAD` genügt, um Erreichbarkeit, Grösse und Stand zu sehen — ohne 24 MB zu
laden.

In [ ]:
print(f'{"Station":<13} {"Stadt":<9} {"Status":<7} {"Groesse":>10}  Stand der Quelle')
for sid, info in STATIONS.items():
    try:
        kopf = session.head(dn.station_url(sid), timeout=30, allow_redirects=True)
        bytes_gross = int(kopf.headers.get('Content-Length', 0))
        print(f'{sid:<13} {info["city"]:<9} {kopf.status_code:<7} '
              f'{bytes_gross / 1e6:>7.1f} MB  {kopf.headers.get("Last-Modified", "-")}')
    except requests.RequestException as fehler:
        print(f'{sid:<13} {info["city"]:<9} nicht erreichbar: {fehler}')

### 6.2 Wie viele Spalten es wirklich sind

Die Kopfzeile steht in den ersten Bytes der Datei, also reicht ein
angeschnittener Download. Die Differenz zwischen den beiden Stationen ist der
Beleg für Feststellung 5.2.

In [ ]:
kopfzeilen = {}
for sid in DEFAULT_STATIONS:
    try:
        antwort = session.get(dn.station_url(sid), timeout=60, stream=True)
        antwort.raise_for_status()
        anfang = next(antwort.iter_content(8192)).decode('utf-8', 'replace')
        antwort.close()
        kopfzeilen[sid] = anfang.splitlines()[0].replace('"', '').split(',')
    except requests.RequestException as fehler:
        print(sid, 'nicht erreichbar:', fehler)

for sid, spalten in kopfzeilen.items():
    print(f'{sid}  {len(spalten):>3} Spalten')

if len(kopfzeilen) == 2:
    (a_id, a), (b_id, b) = kopfzeilen.items()
    print()
    print(f'nur bei {a_id}:', sorted(set(a) - set(b)) or 'keine')
    print(f'nur bei {b_id}:', sorted(set(b) - set(a)) or 'keine')
    print()
    print('gepinnt werden davon:', len(dn.FIELD_TYPES), 'Spalten')
    print('nach _extra wandern :', len(set(a) - set(dn.FIELD_TYPES)), 'Spalten (bei', a_id + ')')

### 6.3 Die Zehntel-Falle vorführen

`read_series` liest die CSV genau so, wie die Ingestion es tut. Der Rohwert
zeigt, was in Bronze landet — und was Silver daraus machen muss.

In [ ]:
sid = DEFAULT_STATIONS[0]
reihe = None
try:
    antwort = session.get(dn.station_url(sid), timeout=120)
    antwort.raise_for_status()
    reihe = dn.read_series(antwort.content, sid)
except requests.RequestException as fehler:
    print('nicht erreichbar:', fehler)

if reihe is not None:
    letzte = reihe.rows[-1]
    print(f'{sid}, letzter Tag der Reihe: {letzte["DATE"]}')
    print()
    print(f'{"Spalte":<6} {"roh":>10}   {"geparst":>9}  Flags')
    for name in dn.MEASUREMENT_COLUMNS:
        roh = letzte[name]
        teiler = 10 if name in ('TMAX', 'TMIN', 'PRCP') else 1
        geparst = int(roh) / teiler if roh.strip() else None
        einheit = '\u00b0C' if name in ('TMAX', 'TMIN') else 'mm'
        print(f'{name:<6} {roh!r:>10}   {geparst!s:>9} {einheit}  '
              f'{letzte[name + dn.ATTRIBUTE_SUFFIX]!r}')
    print()
    print('Achtung: SNOW steht in Millimetern, TMAX/TMIN/PRCP in Zehnteln.')

### 6.4 Der Nachlauf der Quelle

Die Differenz zwischen heute und dem letzten Tag der Reihe ist Feststellung 5.4.

In [ ]:
from datetime import date

if reihe is not None:
    letzter = date.fromisoformat(reihe.last_date)
    print('erster Tag der Reihe:', reihe.first_date)
    print('letzter Tag der Reihe:', reihe.last_date)
    print('heute                :', date.today().isoformat())
    print()
    print('Nachlauf:', (date.today() - letzter).days, 'Tage')
    print()
    print('Ein Join auf Faelle der letzten Tage findet keine Wetterzeile.')

### 6.5 Wann die Messreihe wirklich beginnt

Ab hier wird der geladene Lauf gelesen, nicht das Netz. Ohne Lauf zuerst
Abschnitt 9 ausführen.

Die Tabelle ist Feststellung 5.5: Zeilen gibt es ab 1946, `TMAX` erst ab Ende
1958, und ab den 1960ern ist die Belegung lückenlos.

In [ ]:
import pyarrow.parquet as pq


def neuester_lauf(station: str) -> Path | None:
    muster = f'bronze/dataset={dn.DATASET}/station={station}/run_*/part-0001.parquet'
    treffer = sorted((PROJECT_ROOT / 'data').glob(muster))
    return treffer[-1] if treffer else None


pfade = {sid: neuester_lauf(sid) for sid in DEFAULT_STATIONS}
for sid, pfad in pfade.items():
    print(f'{sid}  {pfad.parent.name if pfad else "kein Lauf - siehe Abschnitt 9"}')

In [ ]:
for sid, pfad in pfade.items():
    if pfad is None:
        continue
    tabelle = pq.read_table(pfad, columns=['DATE', 'TMAX'])
    je_dekade: dict[str, list[int]] = collections.defaultdict(lambda: [0, 0])
    for datum, wert in zip(tabelle.column('DATE').to_pylist(),
                           tabelle.column('TMAX').to_pylist()):
        eintrag = je_dekade[f'{datum[:3]}0er']
        eintrag[0] += 1
        if wert is not None:
            eintrag[1] += 1

    print(f'--- {sid} · {STATIONS[sid]["name"]} ---')
    for dekade in sorted(je_dekade):
        gesamt, belegt = je_dekade[dekade]
        print(f'  {dekade}  TMAX in {belegt:>5,} von {gesamt:>5,} Zeilen  ({belegt / gesamt:5.1%})')
    print()

### 6.6 Was in `_extra` landet

`_extra` ist bei dieser Quelle kein Randfall, sondern die Regel: Jede Zeile trägt
Grössen ausserhalb des gepinnten Schemas. Die Liste zeigt, was dort ohne neuen
Ladelauf abrufbar liegt.

In [ ]:
for sid, pfad in pfade.items():
    if pfad is None:
        continue
    tabelle = pq.read_table(pfad, columns=['_extra'])
    schluessel: collections.Counter[str] = collections.Counter()
    belegt = 0
    for text in tabelle.column('_extra').to_pylist():
        if text:
            belegt += 1
            schluessel.update(json.loads(text).keys())

    print(f'--- {sid} ---')
    print(f'  _extra belegt in {belegt:,} von {tabelle.num_rows:,} Zeilen '
          f'({belegt / tabelle.num_rows:.1%})')
    print(f'  {len(schluessel)} verschiedene Schluessel, haeufigste:')
    for name, anzahl in schluessel.most_common(8):
        print(f'    {name:<20} {anzahl:>7,}')
    print()

### 6.7 Die Qualitätsflags auszählen

Die Positionen von `M,Q,S` und, wo vorhanden, die Beobachtungszeit. Die
Qualitätsposition ist die interessante — und der Grund, alle vier Grössen
auszuzählen statt von zwei auf alle zu schliessen: Bei `SNOW` an O'Hare stehen
vier Beanstandungen (Feststellung 5.6). Die ausgezählte Feldzahl belegt
Feststellung 5.8.

In [ ]:
POSITIONEN = ('M Messung', 'Q Qualitaet', 'S Quelle', 'Beobachtungszeit')

for sid, pfad in pfade.items():
    if pfad is None:
        continue
    spalten = [g + dn.ATTRIBUTE_SUFFIX for g in dn.MEASUREMENT_COLUMNS]
    tabelle = pq.read_table(pfad, columns=spalten)
    print(f'--- {sid} ---')
    for spalte in spalten:
        werte = [t for t in tabelle.column(spalte).to_pylist() if t]
        if not werte:
            print(f'  {spalte:<18} keine Flags')
            continue
        # Die Feldzahl wird ausgezaehlt und nicht unterstellt - sie
        # unterscheidet sich je Groesse (Feststellung 5.8).
        feldzahl = collections.Counter(len(t.split(',')) for t in werte)
        breite = max(feldzahl)
        zaehler = [collections.Counter() for _ in range(breite)]
        for text in werte:
            for position, wert in enumerate(text.split(',')):
                zaehler[position][wert or '(leer)'] += 1
        print(f'  {spalte:<18} Feldzahl {dict(feldzahl)}')
        for position in range(breite):
            name = POSITIONEN[position] if position < len(POSITIONEN) else f'Feld {position + 1}'
            print(f'    {name:<17} {dict(zaehler[position].most_common(4))}')
    print()

### 6.8 Das Manifest als Vollständigkeitsnachweis

Was die Ingestion belegt hat, steht im Manifest — nicht im Notebook und nicht im
Gedächtnis.

In [ ]:
for sid, pfad in pfade.items():
    if pfad is None:
        continue
    manifest = json.loads((pfad.parent / 'manifest.json').read_text(encoding='utf-8'))
    nachweis = manifest['completeness']
    stimmt = lambda a, b: 'ok' if a == b else 'ABWEICHUNG'

    print(f'--- {sid} · {manifest["station_name"]} ({manifest["city"]}) ---')
    print(f'  Lauf            {manifest["run_id"]}')
    print(f'  Abrufdatum      {manifest["ingest_date"]}')
    print(f'  Zeitraum        {manifest["first_observation_date"]} bis '
          f'{manifest["last_observation_date"]}')
    print(f'  Zeilen          {manifest["row_count"]:,}')
    print(f'  Join-Achse      {manifest["join_keys"]}')
    print(f'  Content-Length  {nachweis["content_length"]:,} = empfangen '
          f'{nachweis["downloaded_bytes"]:,}  '
          f'{stimmt(nachweis["content_length"], nachweis["downloaded_bytes"])}')
    print(f'  Zeilen gelesen  {nachweis["csv_rows_read"]:,} = geschrieben '
          f'{nachweis["rows_written"]:,}  '
          f'{stimmt(nachweis["csv_rows_read"], nachweis["rows_written"])}')
    print(f'  Station je Zeile geprueft: {nachweis["station_column_consistent"]}')
    print(f'  Quellspalten    {manifest["source_schema"]["column_count"]}  '
          f'Fingerabdruck {manifest["source_schema"]["fingerprint"][:16]}')
    # Mit diesen beiden fragt der naechste Lauf bedingt an.
    print(f'  ETag            {manifest["query"].get("etag")}')
    print(f'  Last-Modified   {manifest["query"].get("last_modified")}')
    print()

## 7. Datenqualität und Grenzen

**Eine Station ist kein Stadtmittel.** O'Hare liegt am nordwestlichen Stadtrand,
Detroit Metro in Romulus, rund 30 km südwestlich der Innenstadt. Für Temperatur
und Niederschlag ist der Unterschied zur Innenstadt klein, aber nicht null —
städtische Wärmeinseln liegen im Sommer mehrere Grad über dem Umland. Wer das
ernst nimmt, nimmt Midway als zweite Chicagoer Station hinzu und vergleicht.

**Der Zeitstempel ist ein Kalendertag in Ortszeit der Station**, ohne Uhrzeit und
ohne Zeitzonenangabe in den Daten. Ein Tagesmaximum sagt nicht, wann es erreicht
wurde. Für eine Frage nach Tageszeit und Kriminalität reicht GHCN-Daily nicht —
dafür bräuchte es Stundenwerte aus einer anderen Quelle.

**`SNOW` ist Neuschnee, nicht Schneedecke.** Die Schneedecke heisst `SNWD` und
liegt in `_extra`. Wer nach „lag Schnee" fragt, braucht `SNWD`, nicht `SNOW`.

**Fehlend heisst fehlend, nicht null.** Ein leeres CSV-Feld wird in Bronze zu
`NULL` und nicht zu `0`. Das ist bei `PRCP` der entscheidende Unterschied: kein
Messwert ist nicht dasselbe wie kein Niederschlag. Anders als beim ACS mit seinen
`-666666666` gibt es hier keine Sentinel-Werte.

**Die vier Grössen fehlen in den ältesten Zeilen vollständig** (Feststellung
5.5). Eine Auswertung über die ganze Reihe muss das Startjahr je Grösse aus den
Daten bestimmen, nicht aus `first_observation_date`.

**Was Bronze ausdrücklich nicht tut:** parsen, umrechnen, Einheiten normieren,
Flags interpretieren, Lücken füllen, zwischen Stationen mitteln. All das gehört
nach Silver und Gold.

## 8. Entscheidungen für die Wetter-Ingestion

Jede Zeile ist eine Antwort auf eine Eigenschaft der Quelle.

| Entscheidung | Grund |
| --- | --- |
| Partition `dataset=noaa_ghcnd/station=<ID>` | Wetter ist kein Bundesstaats-, sondern ein Stadtmerkmal; Chicago hätte zwei Stationen im selben Staat |
| `join_keys` im Manifest | die Achse ist nicht aus dem Pfad ableitbar (Abschnitt 3) |
| Gepinntes Schema plus `_extra` | 112 bzw. 116 Quellspalten, je Station verschieden (5.1, 5.2) |
| Fingerabdruck **je Station** | verschiedene Spaltenzahl ist normal, eine geänderte für dieselbe Station ist das Signal (5.2) |
| Werte bleiben Text | Zehntel und Auffüllung sind Parse-Aufgaben, und 4.3 verbietet Parsen in Bronze (5.3) |
| `value_encoding` im Manifest | damit Silver die Einheit nicht raten muss (5.3) |
| Jeder Lauf ein vollständiger Schnappschuss | die Quelle liefert kein Delta, nur die ganze Datei |
| Mindestabstand von sieben Tagen | jeder Lauf ist vollständig, ein selteneres Laden erzeugt keine Lücke; täglich kostet 8,3 GB im Jahr statt 1,2 GB (Projektabsprache 3.1) |
| Bedingter Abruf über `If-None-Match` | die Quelle liefert `HTTP 304` mit null Bytes, wenn sie unverändert ist; greift auch bei `--force` |
| `first_`/`last_observation_date` im Manifest | Silver ordnet Schnappschüsse, ohne Parquet zu lesen |
| Station je Zeile geprüft | fängt eine falsch einsortierte Partition, die sonst erst in Silver auffällt (5.7) |
| Leeres Feld wird `NULL` | fehlender Messwert ist nicht `0` (Abschnitt 7) |

**Was bewusst offen bleibt:** Ob Variante C überhaupt kommt (Abschnitt 2.1), ob
Midway dazukommt, und ob `TAVG` oder `SNWD` ins gepinnte Schema wandern. Für die
letzten beiden liegen die Daten schon in Bronze — es kostet einen Eintrag in
`MEASUREMENT_COLUMNS` und keinen neuen Ladelauf.

## 9. Die Ingestion aufrufen

Aus dem Projekt-Root, nicht aus `notebooks/`:

```bash
python -m weather_pipeline.download_noaa
```

Das lädt die beiden Flughafenstationen. Weitere Möglichkeiten:

```bash
# Chicago Midway zusätzlich
python -m weather_pipeline.download_noaa --station USW00014819

# Mindestabstand ignorieren; laedt trotzdem nur, wenn die Quelle neu ist
python -m weather_pipeline.download_noaa --force

# Mindestabstand aufheben
python -m weather_pipeline.download_noaa --min-interval-days 0

# in ein Testverzeichnis, ohne data/ anzufassen
python -m weather_pipeline.download_noaa --data-root /tmp/wetter-test
```

Ergebnis je Station und Lauf:

```text
data/raw/dataset=noaa_ghcnd/station=USW00094846/run_<UTC-Zeitstempel>/
├── USW00094846.csv        # Byte für Byte wie ausgeliefert
└── manifest.json

data/bronze/dataset=noaa_ghcnd/station=USW00094846/run_<UTC-Zeitstempel>/
├── part-0001.parquet      # gepinntes Schema, Werte als Text
└── manifest.json          # zusätzlich derived_from und schema
```

Beide Schichten teilen dieselbe `run_id`. Raw wird zuerst veröffentlicht: ein
Raw-Lauf ohne Bronze-Zwilling kostet eine Wiederholung, nie eine Lücke.

Die Tests laufen ohne Netz:

```bash
python -m pytest tests/weather_pipeline
```

## 10. Offizielle Quellen

| Was | Adresse |
| --- | --- |
| Verzeichnis aller Stationsdateien | <https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/> |
| Stationsliste `ghcnd-stations.txt` | <https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt> |
| Formatbeschreibung `readme.txt` | <https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt> |
| Beschreibung des Datensatzes | <https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily> |

Die Formatbeschreibung ist die Referenz für die Bedeutung der Grössenkürzel und
der Flags. Die Stationsliste enthält Koordinaten, Höhe und Betriebszeitraum
jeder Station — dort steht auch, welche weiteren Stationen in Cook County und
Wayne County messen.

## Appendix A: Das gepinnte Schema

In [ ]:
print(f'{len(dn.SCHEMA)} Spalten:')
for feld in dn.SCHEMA:
    rolle = ''
    if feld.name in dn.IDENTITY_COLUMNS:
        rolle = 'Identitaet'
    elif feld.name == dn.EXTRA_COLUMN:
        rolle = 'alles Uebrige, als JSON je Zeile'
    elif feld.name.endswith(dn.ATTRIBUTE_SUFFIX):
        rolle = 'Qualitaetsflags M,Q,S'
    else:
        rolle = 'Messgroesse'
    print(f'  {feld.name:<20} {str(feld.type):<8} {rolle}')

## Appendix B: Die Flags in `<GRÖSSE>_ATTRIBUTES`

Kommagetrennte Positionen, je einzeln optional und leer, wenn nichts zu melden
ist. `,,0` heisst: kein Messhinweis, keine Beanstandung, Quelle `0`.

| Position | Name | Bedeutung | In diesen Daten beobachtet |
| --- | --- | --- | --- |
| 1 | **M**easurement | Hinweis zur Messung selbst | `T` Spurenmenge (bei `PRCP` rund 4'200 Zeilen je Station), sonst leer |
| 2 | **Q**uality | Beanstandung durch NOAAs Prüfung | leer bei `TMAX`, `TMIN`, `PRCP`; bei O'Hares `SNOW` viermal `I` |
| 3 | **S**ource | Meldeweg, aus dem der Wert stammt | `0`, `W`, `X`, `1`, bei Detroits `SNOW` auch `Z` |
| 4 | Beobachtungszeit | `HHMM`, nur bei manchen Spalten | `2400`, leer, oder der literale Text `null` |

**Die Feldzahl unterscheidet sich je Grösse und je Station** (Feststellung 5.8):
`TMAX` und `TMIN` haben überall drei Felder, `PRCP` überall vier, `SNOW` vier bei
O'Hare und drei bei Detroit Metro. Innerhalb einer Spalte einer Station ist sie
konstant — dort lässt sie sich bestimmen, aber nicht einmal für die Quelle.

Für die Auswertung zählt vor allem Position 2: **Ein Wert dort heisst, dass NOAA
den Messwert für zweifelhaft hält.** Sie ist fast immer leer, mit vier
Ausnahmen bei O'Hares `SNOW` (Abschnitt 6.7 zählt es aus). Die Silver-Tabelle
`weather_day` benennt solche Zeilen mit `SUSPECT_SNOWFALL` und entfernt sie
nicht.

Position 1 ist bei Niederschlag wichtig: `T` steht für eine Spurenmenge — messbar
vorhanden, aber unter der Auflösung. Der Wert erscheint dann als `0` mit `T` als
Messflag, und das ist fachlich etwas anderes als trocken. Wer `PRCP = 0` als
„kein Regen" liest, zählt rund 4'200 Spurenmengen-Tage je Station falsch.

Die vollständige Liste der Kürzel steht in der Formatbeschreibung
(`readme.txt`, Abschnitt III).